1. 把真实的主要代谢物smiles放进pred_results_score.csv

In [35]:
import pandas as pd
from rdkit import Chem
from tqdm import tqdm

In [36]:
import pandas as pd
from rdkit import Chem
from rdkit.DataStructs import TanimotoSimilarity
from rdkit.Chem import AllChem

def get_similarity(smiles1, smiles2):
    mol1 = Chem.MolFromSmiles(smiles1)
    mol2 = Chem.MolFromSmiles(smiles2)
    if mol1 and mol2:
        fp1 = AllChem.GetMorganFingerprintAsBitVect(mol1, 2, nBits=2048)
        fp2 = AllChem.GetMorganFingerprintAsBitVect(mol2, 2, nBits=2048)
        return TanimotoSimilarity(fp1, fp2)
    return 0


In [37]:
# 提取出正确的主要代谢物放到true_mains列
df = pd.read_csv('pred_results_score_new.csv')
true_mains = []
for i in tqdm(range(len(df))):
    predicts = df.iloc[i,2]
    try:
        predicts = predicts.split('|')
    except:
        predicts = ['None']
    top_n = df.iloc[i,3]
    if len(top_n) == 1:
        top_n = int(top_n)
        true_main = predicts[top_n-1].split(' ')[0]
        true_main = [true_main]
    else:
        true_main = top_n.split(',')
    true_main = '|'.join(true_main)
    true_mains.append(true_main)
df['true_mains'] = true_mains
df

100%|██████████| 30/30 [00:00<00:00, 15110.98it/s]


,name,substrate,predict,top_n,true_mains
0,1,CC(C)(C)[C@H](NC(=O)C(F)(F)F)C(=O)N1CC2(C[C@H]...,CC(C)(C)[C@H](NC(=O)C(F)(F)F)C(=O)N1CC2(C[C@H]...,2,CC(C)(C)[C@H](NC(=O)C(F)(F)F)C(=O)N1CC2(C[C@H]...
1,2,CC[C@@H](C(=O)Nc1ccc(C(N)=O)c(F)c1)n1cc(OC)c(-...,CC[C@@H](C(=O)Nc1ccc(C(N)=O)c(F)c1)n1cc(O)c(-c...,3,CC[C@@H](C(=O)O)n1cc(OC)c(-c2cc(Cl)ccc2-n2cc(C...
2,3,CCCCCc1cc(O)c2c(c1)OC(C)(C)C1CCC(C)CC21,CCCCCc1cc(O)c2c(c1)OC(C)(C)C1CCC(CO)CC21 0.57|...,1,CCCCCc1cc(O)c2c(c1)OC(C)(C)C1CCC(CO)CC21
3,4,CC(=O)Nc1ccc(C)c(C(=O)N[C@H](C)c2cccc3ccccc23)c1,Cc1ccc(N)cc1C(=O)N[C@H](C)c1cccc2ccccc12 0.83|...,1,Cc1ccc(N)cc1C(=O)N[C@H](C)c1cccc2ccccc12
4,5,Cc1ccc(N)cc1C(=O)N[C@H](C)c1cccc2ccccc12,CC(=O)Nc1ccc(C)c(C(=O)N[C@H](C)c2cccc3ccccc23)...,2,Cc1cc(O)c(N)cc1C(=O)N[C@H](C)c1cccc2ccccc12
5,6,Cn1cc(Nc2nccc(N3C[C@H]4CC[C@@H](C3)N4C(=O)[C@@...,O=C([C@@H]1CC1(F)F)N1[C@H]2CC[C@@H]1CN(c1ccnc(...,4,Nc1nccc(N2C[C@@H]3CC[C@H](C2)N3C(=O)[C@@H]2CC2...
6,7,C=CC(=O)Nc1cc(Nc2ncc(C(=O)OC(C)C)c(-c3cn(C)c4c...,C=CC(=O)Nc1cc(Nc2ncc(C(=O)OC(C)C)c(-c3c[nH]c4c...,2,C=CC(=O)Nc1cc(Nc2ncc(C(=O)OC(C)C)c(-c3cn(C)c4c...
7,8,Cc1ccc([C@@H]2O[C@H](CO)[C@@H](O)[C@H](O)[C@H]...,OCc1ccc([C@@H]2O[C@H](CO)[C@@H](O)[C@H](O)[C@H...,3,Cc1ccc([C@@H]2O[C@H](CO)[C@@H](O)[C@H](OC3O[C@...
8,10,O=C(Nc1ccc(OC(F)(F)F)cc1)c1sccc1NCc1ccnc2ccccc12,Nc1ccsc1C(=O)Nc1ccc(OC(F)(F)F)cc1 0.73|O=C(Nc1...,2,O=C(Nc1ccc(OC(F)(F)F)cc1)c1sccc1NCc1cc(=O)[nH]...
9,11,NC1(C(=O)N[C@@H](CCO)c2ccc(Cl)cc2)CCN(c2ncnc3[...,NC1(C(=O)N[C@@H](CCOC2O[C@H](C(=O)O)[C@@H](O)[...,1,NC1(C(=O)N[C@@H](CCOC2O[C@H](C(=O)O)[C@@H](O)[...


In [38]:
# 将substrate和true_mains列的smiles标准化
def canonical_smiles(smi):
    mol = Chem.MolFromSmiles(smi)
    smi_new = Chem.MolToSmiles(mol)
    return smi_new

substrates = df['substrate'].to_list()
true_mains = df['true_mains'].to_list()
substrates_cano = [canonical_smiles(i) for i in substrates]
true_mains_cano = [canonical_smiles(i) for i in true_mains]

df['substrate'] = substrates_cano
df['true_mains'] = true_mains_cano

df

,name,substrate,predict,top_n,true_mains
0,1,CC(C)(C)[C@H](NC(=O)C(F)(F)F)C(=O)N1CC2(C[C@H]...,CC(C)(C)[C@H](NC(=O)C(F)(F)F)C(=O)N1CC2(C[C@H]...,2,CC(C)(C)[C@H](NC(=O)C(F)(F)F)C(=O)N1CC2(C[C@H]...
1,2,CC[C@@H](C(=O)Nc1ccc(C(N)=O)c(F)c1)n1cc(OC)c(-...,CC[C@@H](C(=O)Nc1ccc(C(N)=O)c(F)c1)n1cc(O)c(-c...,3,CC[C@@H](C(=O)O)n1cc(OC)c(-c2cc(Cl)ccc2-n2cc(C...
2,3,CCCCCc1cc(O)c2c(c1)OC(C)(C)C1CCC(C)CC21,CCCCCc1cc(O)c2c(c1)OC(C)(C)C1CCC(CO)CC21 0.57|...,1,CCCCCc1cc(O)c2c(c1)OC(C)(C)C1CCC(CO)CC21
3,4,CC(=O)Nc1ccc(C)c(C(=O)N[C@H](C)c2cccc3ccccc23)c1,Cc1ccc(N)cc1C(=O)N[C@H](C)c1cccc2ccccc12 0.83|...,1,Cc1ccc(N)cc1C(=O)N[C@H](C)c1cccc2ccccc12
4,5,Cc1ccc(N)cc1C(=O)N[C@H](C)c1cccc2ccccc12,CC(=O)Nc1ccc(C)c(C(=O)N[C@H](C)c2cccc3ccccc23)...,2,Cc1cc(O)c(N)cc1C(=O)N[C@H](C)c1cccc2ccccc12
5,6,Cn1cc(Nc2nccc(N3C[C@H]4CC[C@@H](C3)N4C(=O)[C@@...,O=C([C@@H]1CC1(F)F)N1[C@H]2CC[C@@H]1CN(c1ccnc(...,4,Nc1nccc(N2C[C@H]3CC[C@@H](C2)N3C(=O)[C@@H]2CC2...
6,7,C=CC(=O)Nc1cc(Nc2ncc(C(=O)OC(C)C)c(-c3cn(C)c4c...,C=CC(=O)Nc1cc(Nc2ncc(C(=O)OC(C)C)c(-c3c[nH]c4c...,2,C=CC(=O)Nc1cc(Nc2ncc(C(=O)OC(C)C)c(-c3cn(C)c4c...
7,8,Cc1ccc([C@@H]2O[C@H](CO)[C@@H](O)[C@H](O)[C@H]...,OCc1ccc([C@@H]2O[C@H](CO)[C@@H](O)[C@H](O)[C@H...,3,Cc1ccc([C@@H]2O[C@H](CO)[C@@H](O)[C@H](OC3O[C@...
8,10,O=C(Nc1ccc(OC(F)(F)F)cc1)c1sccc1NCc1ccnc2ccccc12,Nc1ccsc1C(=O)Nc1ccc(OC(F)(F)F)cc1 0.73|O=C(Nc1...,2,O=C(Nc1ccc(OC(F)(F)F)cc1)c1sccc1NCc1cc(=O)[nH]...
9,11,NC1(C(=O)N[C@@H](CCO)c2ccc(Cl)cc2)CCN(c2ncnc3[...,NC1(C(=O)N[C@@H](CCOC2O[C@H](C(=O)O)[C@@H](O)[...,1,NC1(C(=O)N[C@@H](CCOC2O[C@H](C(=O)O)[C@@H](O)[...


In [39]:
df.to_csv('pred_results_substrate_truemains.csv',index=None)

In [6]:
# df.to_pickle('pred_results_substrate_truemains.pickle')

In [40]:
# 建立一个真实的dict_sub2main
df = pd.read_csv('pred_results_substrate_truemains.csv')
substrate_cano = df['substrate'].to_list()
true_mains_cano = df['true_mains'].to_list()

dict_sub2main = dict(zip(substrate_cano,true_mains_cano))
dict_sub2main

{'CC(C)(C)[C@H](NC(=O)C(F)(F)F)C(=O)N1CC2(C[C@H]1C(=O)N[C@H](C#N)C[C@@H]1CCNC1=O)SCCS2': 'CC(C)(C)[C@H](NC(=O)C(F)(F)F)C(=O)N1CC2(C[C@H]1C(=O)N[C@@H](C[C@@H]1CCNC1=O)C(=O)O)SCCS2',
 'CC[C@@H](C(=O)Nc1ccc(C(N)=O)c(F)c1)n1cc(OC)c(-c2cc(Cl)ccc2-n2cc(C(F)(F)F)nn2)cc1=O': 'CC[C@@H](C(=O)O)n1cc(OC)c(-c2cc(Cl)ccc2-n2cc(C(F)(F)F)nn2)cc1=O',
 'CCCCCc1cc(O)c2c(c1)OC(C)(C)C1CCC(C)CC21': 'CCCCCc1cc(O)c2c(c1)OC(C)(C)C1CCC(CO)CC21',
 'CC(=O)Nc1ccc(C)c(C(=O)N[C@H](C)c2cccc3ccccc23)c1': 'Cc1ccc(N)cc1C(=O)N[C@H](C)c1cccc2ccccc12',
 'Cc1ccc(N)cc1C(=O)N[C@H](C)c1cccc2ccccc12': 'Cc1cc(O)c(N)cc1C(=O)N[C@H](C)c1cccc2ccccc12',
 'Cn1cc(Nc2nccc(N3C[C@H]4CC[C@@H](C3)N4C(=O)[C@@H]3CC3(F)F)n2)cn1': 'Nc1nccc(N2C[C@H]3CC[C@@H](C2)N3C(=O)[C@@H]2CC2(F)F)n1',
 'C=CC(=O)Nc1cc(Nc2ncc(C(=O)OC(C)C)c(-c3cn(C)c4ccccc34)n2)c(OC)cc1N(C)CCN(C)C': 'C=CC(=O)Nc1cc(Nc2ncc(C(=O)OC(C)C)c(-c3cn(C)c4ccccc34)n2)c(OC)cc1N(C)CCNC',
 'Cc1ccc([C@@H]2O[C@H](CO)[C@@H](O)[C@H](O)[C@H]2O)cc1Cc1ccc(-c2ccc(F)cc2)s1': 'Cc1ccc([C@@H]2O[C@H](CO)[C@

2. 整理 SyGMa 的结果

In [44]:
# 找到SyGMa_main.csv中每个底物对应的预测值正确的序号true_rank，并建立一个dict_sygma_sub2truerank存放
df_sygma = pd.read_csv('SyGMa_predict.csv')
dict_sub2truerank = {}
for i in tqdm(range(len(df_sygma))):
    substrate = df_sygma.iloc[i,0]
    predicts = df_sygma.iloc[i,1]
    predicts = predicts.split('|')

    # 底物要能对应上
    substrate = canonical_smiles(substrate)
    true_mains_sygma = dict_sub2main[substrate]

    # 每个底物只看sygma预测出的top_5
    predicts_top_5 = predicts[:5]
    predicts_top_5 = list(map(canonical_smiles,predicts_top_5))


    # for j in range(len(predicts_top_5)):
    #     predict_top_5 = predicts_top_5[j]
    #     for x in true_mains_sygma:
    #         if get_similarity(predict_top_5,x) == 1:
    #             true_rank.append(j+1)

    for j, predict_top_5_new in enumerate(predicts_top_5):
        # 检查 predict_top_5 是否与任何 true_mains_sygma 匹配
        if get_similarity(predict_top_5_new, true_mains_sygma) == 1:
            dict_sub2truerank[substrate] = j+1
            # true_rank.append(j+1)
            break

substrates_sygma = df_sygma['smi'].to_list()
substrates_sygma_cano = list(map(canonical_smiles,substrates_sygma))
sygma_ranks = [dict_sub2truerank.get(i,0) for i in substrates_sygma_cano]
df_sygma['smi'] = substrates_sygma_cano
df_sygma['sygma_rank'] = sygma_ranks
df_sygma
# dict_sygma_sub2truerank = dict(zip(substrates_sygma_cano,true_rank))
# dict_sygma_sub2truerank

  0%|          | 0/30 [00:00<?, ?it/s]

100%|██████████| 30/30 [00:00<00:00, 234.66it/s]


,smi,extracted_strings,sygma_rank
0,CN(C(=O)Cc1cccs1)[C@@H](Cc1c[nH]c2ccccc12)C(=O...,CN(C(=O)Cc1cccs1)[C@@H](Cc1c[nH]c2ccccc12)C(=O...,3
1,C/C=C/C#CC#CC(O)C(/C=C/CCCO)OC1OC(COC2OC(CO)C(...,C/C=C/C#CC#CC(O)C(/C=C/CCCO)OC1OC(COC2OC(CO)C(...,0
2,C=CC(=O)Nc1cc(Nc2ncc(C(=O)OC(C)C)c(-c3cn(C)c4c...,C=CC(=O)Nc1cc(Nc2ncc(C(=O)OC(C)C)c(-c3cn(C)c4c...,2
3,CC(=O)Nc1ccc(C)c(C(=O)N[C@H](C)c2cccc3ccccc23)c1,CC(=O)Nc1ccc(C)c(C(=O)N[C@H](C)c2cccc3ccccc23)...,2
4,CCN(CC)CCc1c[nH]c2cccc(OC(C)=O)c12,CCN(CC)CCc1c[nH]c2cccc(OC(C)=O)c12|CCN(CC)CCc1...,2
5,Cc1ccc(N)cc1C(=O)N[C@H](C)c1cccc2ccccc12,Cc1ccc(N)cc1C(=O)N[C@H](C)c1cccc2ccccc12|CC(=O...,0
6,Cc1ccc([C@@H]2O[C@H](CO)[C@@H](O)[C@H](O)[C@H]...,Cc1ccc([C@@H]2O[C@H](CO)[C@@H](O)[C@H](O)[C@H]...,0
7,Cc1ncc(S(=O)(=O)N2CCC(c3cn4ncnc4cc3Cl)CC2)s1,Cc1ncc(S(=O)(=O)N2CCC(c3cn4ncnc4cc3Cl)CC2)s1|O...,2
8,CCCCCc1cc(O)c2c(c1)OC(C)(C)C1CCC(C)CC21,CCCCCc1cc(O)c2c(c1)OC(C)(C)C1CCC(C)CC21|CCCCCc...,0
9,CC[C@H](C)C(=O)N1CCC(NC(=O)Nc2ccc(OC(F)(F)F)c(...,CC[C@H](C)C(=O)N1CCC(NC(=O)Nc2ccc(OC(F)(F)F)c(...,0


In [45]:
dict_sygma_sub2truerank = pd.Series(df_sygma.sygma_rank.values, index=df_sygma.smi).to_dict()
dict_sygma_sub2truerank

{'CN(C(=O)Cc1cccs1)[C@@H](Cc1c[nH]c2ccccc12)C(=O)N[C@@](C)(Cc1ccccc1)C(=O)N[C@@H](CCCNC(=N)N)C(=O)N1CCC[C@H]1C(=O)N[C@@H](CCCNC(=N)N)C(N)=O': 3,
 'C/C=C/C#CC#CC(O)C(/C=C/CCCO)OC1OC(COC2OC(CO)C(O)C(O)C2O)C(O)C(O)C1O': 0,
 'C=CC(=O)Nc1cc(Nc2ncc(C(=O)OC(C)C)c(-c3cn(C)c4ccccc34)n2)c(OC)cc1N(C)CCN(C)C': 2,
 'CC(=O)Nc1ccc(C)c(C(=O)N[C@H](C)c2cccc3ccccc23)c1': 2,
 'CCN(CC)CCc1c[nH]c2cccc(OC(C)=O)c12': 2,
 'Cc1ccc(N)cc1C(=O)N[C@H](C)c1cccc2ccccc12': 0,
 'Cc1ccc([C@@H]2O[C@H](CO)[C@@H](O)[C@H](O)[C@H]2O)cc1Cc1ccc(-c2ccc(F)cc2)s1': 0,
 'Cc1ncc(S(=O)(=O)N2CCC(c3cn4ncnc4cc3Cl)CC2)s1': 2,
 'CCCCCc1cc(O)c2c(c1)OC(C)(C)C1CCC(C)CC21': 0,
 'CC[C@H](C)C(=O)N1CCC(NC(=O)Nc2ccc(OC(F)(F)F)c(F)c2)CC1': 0,
 'Cn1cc(Nc2nccc(N3C[C@H]4CC[C@@H](C3)N4C(=O)[C@@H]3CC3(F)F)n2)cn1': 0,
 'CCN(CC)C(=O)C1C=C2c3cccc4c3c(cn4C(=O)C3CC3)CC2N(C)C1': 0,
 'COc1c([C@H]2C(C(=O)Nc3ccnc(C(N)=O)c3)O[C@@](C)(C(F)(F)F)[C@H]2C)ccc(F)c1F': 3,
 'COc1ccc(CCCCOC2OC(CO)[C@@H](O)[C@H](O)[C@H]2O)cc1': 2,
 'COc1ccc2c(Oc3ccc(C4(C(=O)NO)CCOCC4)cc

In [46]:
# 在df中增加一列存放true_rank_sygma
df = pd.read_csv('pred_results_substrate_truemains.csv')
substrate_cano = df['substrate'].to_list()
true_rank_sygma = [dict_sygma_sub2truerank[i] for i in substrates_cano]
df['true_rank_sygma'] = true_rank_sygma
df


,name,substrate,predict,top_n,true_mains,true_rank_sygma
0,1,CC(C)(C)[C@H](NC(=O)C(F)(F)F)C(=O)N1CC2(C[C@H]...,CC(C)(C)[C@H](NC(=O)C(F)(F)F)C(=O)N1CC2(C[C@H]...,2,CC(C)(C)[C@H](NC(=O)C(F)(F)F)C(=O)N1CC2(C[C@H]...,0
1,2,CC[C@@H](C(=O)Nc1ccc(C(N)=O)c(F)c1)n1cc(OC)c(-...,CC[C@@H](C(=O)Nc1ccc(C(N)=O)c(F)c1)n1cc(O)c(-c...,3,CC[C@@H](C(=O)O)n1cc(OC)c(-c2cc(Cl)ccc2-n2cc(C...,5
2,3,CCCCCc1cc(O)c2c(c1)OC(C)(C)C1CCC(C)CC21,CCCCCc1cc(O)c2c(c1)OC(C)(C)C1CCC(CO)CC21 0.57|...,1,CCCCCc1cc(O)c2c(c1)OC(C)(C)C1CCC(CO)CC21,0
3,4,CC(=O)Nc1ccc(C)c(C(=O)N[C@H](C)c2cccc3ccccc23)c1,Cc1ccc(N)cc1C(=O)N[C@H](C)c1cccc2ccccc12 0.83|...,1,Cc1ccc(N)cc1C(=O)N[C@H](C)c1cccc2ccccc12,2
4,5,Cc1ccc(N)cc1C(=O)N[C@H](C)c1cccc2ccccc12,CC(=O)Nc1ccc(C)c(C(=O)N[C@H](C)c2cccc3ccccc23)...,2,Cc1cc(O)c(N)cc1C(=O)N[C@H](C)c1cccc2ccccc12,0
5,6,Cn1cc(Nc2nccc(N3C[C@H]4CC[C@@H](C3)N4C(=O)[C@@...,O=C([C@@H]1CC1(F)F)N1[C@H]2CC[C@@H]1CN(c1ccnc(...,4,Nc1nccc(N2C[C@H]3CC[C@@H](C2)N3C(=O)[C@@H]2CC2...,0
6,7,C=CC(=O)Nc1cc(Nc2ncc(C(=O)OC(C)C)c(-c3cn(C)c4c...,C=CC(=O)Nc1cc(Nc2ncc(C(=O)OC(C)C)c(-c3c[nH]c4c...,2,C=CC(=O)Nc1cc(Nc2ncc(C(=O)OC(C)C)c(-c3cn(C)c4c...,2
7,8,Cc1ccc([C@@H]2O[C@H](CO)[C@@H](O)[C@H](O)[C@H]...,OCc1ccc([C@@H]2O[C@H](CO)[C@@H](O)[C@H](O)[C@H...,3,Cc1ccc([C@@H]2O[C@H](CO)[C@@H](O)[C@H](OC3O[C@...,0
8,10,O=C(Nc1ccc(OC(F)(F)F)cc1)c1sccc1NCc1ccnc2ccccc12,Nc1ccsc1C(=O)Nc1ccc(OC(F)(F)F)cc1 0.73|O=C(Nc1...,2,O=C(Nc1ccc(OC(F)(F)F)cc1)c1sccc1NCc1cc(=O)[nH]...,0
9,11,NC1(C(=O)N[C@@H](CCO)c2ccc(Cl)cc2)CCN(c2ncnc3[...,NC1(C(=O)N[C@@H](CCOC2O[C@H](C(=O)O)[C@@H](O)[...,1,NC1(C(=O)N[C@@H](CCOC2O[C@H](C(=O)O)[C@@H](O)[...,5


3. 整理 MetaReact 的结果

In [53]:
# 找到MetaReact_main.csv中每个底物对应的预测值正确的序号true_rank，并建立一个dict_metareact_sub2truerank存放
df_metareact = pd.read_csv('MetaReact_seed42.csv')
dict_metareact_sub2truerank = {}
for i in tqdm(range(len(df_metareact))):
    substrate = df_metareact.iloc[i,1]
    predicts = df_metareact.iloc[i,2]
    predicts = predicts.split('|')

    # 底物要能对应上
    substrate = canonical_smiles(substrate)
    true_mains_metareact = dict_sub2main[substrate]

    # 每个底物只看sygma预测出的top_5
    predicts_top_5 = predicts[:5]
    predicts_top_5 = list(map(canonical_smiles,predicts_top_5))


    for j, predict_top_5_new in enumerate(predicts_top_5):
        # 检查 predict_top_5 是否与任何 true_mains_sygma 匹配
        if get_similarity(predict_top_5_new, true_mains_metareact) == 1:
            dict_metareact_sub2truerank[substrate] = j+1
            break

substrates_metareact = df_metareact['substrate'].to_list()
substrates_metareact_cano = list(map(canonical_smiles,substrates_metareact))
metareact_rank = [dict_metareact_sub2truerank.get(i,0) for i in substrates_metareact_cano]
df_metareact['substrate'] = substrates_metareact_cano
df_metareact['metareact_rank'] = metareact_rank
df_metareact

100%|██████████| 30/30 [00:00<00:00, 285.35it/s]


,name,substrate,predict,metareact_rank
0,1,CC(C)(C)[C@H](NC(=O)C(F)(F)F)C(=O)N1CC2(C[C@H]...,CC(C)(C)[C@H](NC(=O)C(F)(F)F)C(=O)N1CC2(C[C@H]...,0
1,2,CC[C@@H](C(=O)Nc1ccc(C(N)=O)c(F)c1)n1cc(OC)c(-...,CC[C@@H](C(=O)NC1=CC=C(C(=O)O)C(F)=C1)N1C=C(OC...,2
2,3,CCCCCc1cc(O)c2c(c1)OC(C)(C)C1CCC(C)CC21,CC(O)CCCC1=CC(O)=C2C(=C1)OC(C)(C)C1CCC(C)CC21|...,2
3,4,CC(=O)Nc1ccc(C)c(C(=O)N[C@H](C)c2cccc3ccccc23)c1,CC1=CC=C(N)C=C1C(=O)N[C@H](C)C1=CC=CC2=CC=CC=C...,1
4,5,Cc1ccc(N)cc1C(=O)N[C@H](C)c1cccc2ccccc12,CC(=O)NC1=CC=C(C)C(C(=O)N[C@H](C)C2=CC=CC3=CC=...,3
5,6,Cn1cc(Nc2nccc(N3C[C@H]4CC[C@@H](C3)N4C(=O)[C@@...,O=C([C@@H]1CC1(F)F)N1[C@H]2CC[C@@H]1CN(C1=CC=N...,3
6,7,C=CC(=O)Nc1cc(Nc2ncc(C(=O)OC(C)C)c(-c3cn(C)c4c...,C=CC(=O)NC1=CC(NC2=NC=C(C(=O)O)C(C3=CN(C)C4=CC...,2
7,8,Cc1ccc([C@@H]2O[C@H](CO)[C@@H](O)[C@H](O)[C@H]...,OCC1=CC=C([C@@H]2O[C@H](CO)[C@@H](O)[C@H](O)[C...,4
8,10,O=C(Nc1ccc(OC(F)(F)F)cc1)c1sccc1NCc1ccnc2ccccc12,NC1=C(C(=O)NC2=CC=C(OC(F)(F)F)C=C2)SC=C1|O=C(N...,3
9,11,NC1(C(=O)N[C@@H](CCO)c2ccc(Cl)cc2)CCN(c2ncnc3[...,NC1(C(=O)N[C@@H](CCOC2O[C@H](C(=O)O)[C@@H](O)[...,1


In [54]:
dict_metareact_sub2truerank = pd.Series(df_metareact.metareact_rank.values, index=df_metareact.substrate).to_dict()
dict_metareact_sub2truerank

{'CC(C)(C)[C@H](NC(=O)C(F)(F)F)C(=O)N1CC2(C[C@H]1C(=O)N[C@H](C#N)C[C@@H]1CCNC1=O)SCCS2': 0,
 'CC[C@@H](C(=O)Nc1ccc(C(N)=O)c(F)c1)n1cc(OC)c(-c2cc(Cl)ccc2-n2cc(C(F)(F)F)nn2)cc1=O': 2,
 'CCCCCc1cc(O)c2c(c1)OC(C)(C)C1CCC(C)CC21': 2,
 'CC(=O)Nc1ccc(C)c(C(=O)N[C@H](C)c2cccc3ccccc23)c1': 1,
 'Cc1ccc(N)cc1C(=O)N[C@H](C)c1cccc2ccccc12': 3,
 'Cn1cc(Nc2nccc(N3C[C@H]4CC[C@@H](C3)N4C(=O)[C@@H]3CC3(F)F)n2)cn1': 3,
 'C=CC(=O)Nc1cc(Nc2ncc(C(=O)OC(C)C)c(-c3cn(C)c4ccccc34)n2)c(OC)cc1N(C)CCN(C)C': 2,
 'Cc1ccc([C@@H]2O[C@H](CO)[C@@H](O)[C@H](O)[C@H]2O)cc1Cc1ccc(-c2ccc(F)cc2)s1': 4,
 'O=C(Nc1ccc(OC(F)(F)F)cc1)c1sccc1NCc1ccnc2ccccc12': 3,
 'NC1(C(=O)N[C@@H](CCO)c2ccc(Cl)cc2)CCN(c2ncnc3[nH]ccc23)CC1': 1,
 'CCOc1ccc(NC(=S)N(CCO)Cc2cc3cc(C)cc(C)c3[nH]c2=O)cc1': 0,
 'COc1ccc2c(Oc3ccc(C4(C(=O)NO)CCOCC4)cc3)ccnc2n1': 4,
 'C/C=C/C#CC#CC(O)C(/C=C/CCCO)OC1OC(COC2OC(CO)C(O)C(O)C2O)C(O)C(O)C1O': 1,
 'Cc1ncc(S(=O)(=O)N2CCC(c3cn4ncnc4cc3Cl)CC2)s1': 1,
 'COc1ccc(CCCCOC2OC(CO)[C@@H](O)[C@H](O)[C@H]2O)cc1': 1,
 'O=C(O)C1OC

In [55]:
# 在df中增加一列存放true_rank_metareact
substrate_cano = df['substrate'].to_list()
true_rank_metareact = [dict_metareact_sub2truerank[i] for i in substrates_cano]
df['true_rank_metareactseed42'] = true_rank_metareact
df


,name,substrate,predict,top_n,true_mains,true_rank_sygma,true_rank_metareactseed42
0,1,CC(C)(C)[C@H](NC(=O)C(F)(F)F)C(=O)N1CC2(C[C@H]...,CC(C)(C)[C@H](NC(=O)C(F)(F)F)C(=O)N1CC2(C[C@H]...,2,CC(C)(C)[C@H](NC(=O)C(F)(F)F)C(=O)N1CC2(C[C@H]...,0,0
1,2,CC[C@@H](C(=O)Nc1ccc(C(N)=O)c(F)c1)n1cc(OC)c(-...,CC[C@@H](C(=O)Nc1ccc(C(N)=O)c(F)c1)n1cc(O)c(-c...,3,CC[C@@H](C(=O)O)n1cc(OC)c(-c2cc(Cl)ccc2-n2cc(C...,5,2
2,3,CCCCCc1cc(O)c2c(c1)OC(C)(C)C1CCC(C)CC21,CCCCCc1cc(O)c2c(c1)OC(C)(C)C1CCC(CO)CC21 0.57|...,1,CCCCCc1cc(O)c2c(c1)OC(C)(C)C1CCC(CO)CC21,0,2
3,4,CC(=O)Nc1ccc(C)c(C(=O)N[C@H](C)c2cccc3ccccc23)c1,Cc1ccc(N)cc1C(=O)N[C@H](C)c1cccc2ccccc12 0.83|...,1,Cc1ccc(N)cc1C(=O)N[C@H](C)c1cccc2ccccc12,2,1
4,5,Cc1ccc(N)cc1C(=O)N[C@H](C)c1cccc2ccccc12,CC(=O)Nc1ccc(C)c(C(=O)N[C@H](C)c2cccc3ccccc23)...,2,Cc1cc(O)c(N)cc1C(=O)N[C@H](C)c1cccc2ccccc12,0,3
5,6,Cn1cc(Nc2nccc(N3C[C@H]4CC[C@@H](C3)N4C(=O)[C@@...,O=C([C@@H]1CC1(F)F)N1[C@H]2CC[C@@H]1CN(c1ccnc(...,4,Nc1nccc(N2C[C@H]3CC[C@@H](C2)N3C(=O)[C@@H]2CC2...,0,3
6,7,C=CC(=O)Nc1cc(Nc2ncc(C(=O)OC(C)C)c(-c3cn(C)c4c...,C=CC(=O)Nc1cc(Nc2ncc(C(=O)OC(C)C)c(-c3c[nH]c4c...,2,C=CC(=O)Nc1cc(Nc2ncc(C(=O)OC(C)C)c(-c3cn(C)c4c...,2,2
7,8,Cc1ccc([C@@H]2O[C@H](CO)[C@@H](O)[C@H](O)[C@H]...,OCc1ccc([C@@H]2O[C@H](CO)[C@@H](O)[C@H](O)[C@H...,3,Cc1ccc([C@@H]2O[C@H](CO)[C@@H](O)[C@H](OC3O[C@...,0,4
8,10,O=C(Nc1ccc(OC(F)(F)F)cc1)c1sccc1NCc1ccnc2ccccc12,Nc1ccsc1C(=O)Nc1ccc(OC(F)(F)F)cc1 0.73|O=C(Nc1...,2,O=C(Nc1ccc(OC(F)(F)F)cc1)c1sccc1NCc1cc(=O)[nH]...,0,3
9,11,NC1(C(=O)N[C@@H](CCO)c2ccc(Cl)cc2)CCN(c2ncnc3[...,NC1(C(=O)N[C@@H](CCOC2O[C@H](C(=O)O)[C@@H](O)[...,1,NC1(C(=O)N[C@@H](CCOC2O[C@H](C(=O)O)[C@@H](O)[...,5,1


In [58]:
# 找到MetaReact_main.csv中每个底物对应的预测值正确的序号true_rank，并建立一个dict_metareact_sub2truerank存放
df_metareact8079 = pd.read_csv('MetaReact_seed8079.csv')
dict_metareact8079_sub2truerank = {}
for i in tqdm(range(len(df_metareact8079))):
    substrate = df_metareact8079.iloc[i,1]
    predicts = df_metareact8079.iloc[i,2]
    predicts = predicts.split('|')

    # 底物要能对应上
    substrate = canonical_smiles(substrate)
    true_mains_metareact8079 = dict_sub2main[substrate]

    # 每个底物只看sygma预测出的top_5
    predicts_top_5 = predicts[:5]
    predicts_top_5 = list(map(canonical_smiles,predicts_top_5))


    for j, predict_top_5_new in enumerate(predicts_top_5):
        # 检查 predict_top_5 是否与任何 true_mains_sygma 匹配
        if get_similarity(predict_top_5_new, true_mains_metareact8079) == 1:
            dict_metareact8079_sub2truerank[substrate] = j+1
            break

substrates_metareact8079 = df_metareact8079['substrate'].to_list()
substrates_metareact8079_cano = list(map(canonical_smiles,substrates_metareact8079))
metareact8079_rank = [dict_metareact8079_sub2truerank.get(i,0) for i in substrates_metareact8079_cano]
df_metareact8079['substrate'] = substrates_metareact8079_cano
df_metareact8079['metareact_rank'] = metareact8079_rank
df_metareact8079

  0%|          | 0/30 [00:00<?, ?it/s]

100%|██████████| 30/30 [00:00<00:00, 266.06it/s]


,name,substrate,predict,metareact_rank
0,1,CC(C)(C)[C@H](NC(=O)C(F)(F)F)C(=O)N1CC2(C[C@H]...,CC(C)(C)[C@H](NC(=O)C(F)(F)F)C(=O)N1CC2(C[C@H]...,5
1,2,CC[C@@H](C(=O)Nc1ccc(C(N)=O)c(F)c1)n1cc(OC)c(-...,CC[C@@H](C(=O)NC1=CC=C(C(=O)O)C(F)=C1)N1C=C(OC...,2
2,3,CCCCCc1cc(O)c2c(c1)OC(C)(C)C1CCC(C)CC21,CCCCCC1=CC(O)=C2C(=C1)OC(C)(C)C1CCC(CO)CC21|CC...,1
3,4,CC(=O)Nc1ccc(C)c(C(=O)N[C@H](C)c2cccc3ccccc23)c1,CC1=CC=C(N)C=C1C(=O)N[C@H](C)C1=CC=CC2=CC=CC=C...,1
4,5,Cc1ccc(N)cc1C(=O)N[C@H](C)c1cccc2ccccc12,CC(=O)NC1=CC=C(C)C(C(=O)N[C@H](C)C2=CC=CC3=CC=...,2
5,6,Cn1cc(Nc2nccc(N3C[C@H]4CC[C@@H](C3)N4C(=O)[C@@...,O=C([C@@H]1CC1(F)F)N1[C@H]2CC[C@@H]1CN(C1=CC=N...,5
6,7,C=CC(=O)Nc1cc(Nc2ncc(C(=O)OC(C)C)c(-c3cn(C)c4c...,C=CC(=O)NC1=CC(NC2=NC=C(C(=O)OC(C)C)C(C3=CNC4=...,3
7,8,Cc1ccc([C@@H]2O[C@H](CO)[C@@H](O)[C@H](O)[C@H]...,OCC1=CC=C([C@@H]2O[C@H](CO)[C@@H](O)[C@H](O)[C...,0
8,10,O=C(Nc1ccc(OC(F)(F)F)cc1)c1sccc1NCc1ccnc2ccccc12,NC1=C(C(=O)NC2=CC=C(OC(F)(F)F)C=C2)SC=C1|O=C(N...,5
9,11,NC1(C(=O)N[C@@H](CCO)c2ccc(Cl)cc2)CCN(c2ncnc3[...,NC1(C(=O)N[C@@H](CCO)C2=CC=C(Cl)C=C2)CCN(C2=NC...,2


In [59]:
dict_metareact8079_sub2truerank = pd.Series(df_metareact8079.metareact_rank.values, index=df_metareact8079.substrate).to_dict()
dict_metareact8079_sub2truerank

{'CC(C)(C)[C@H](NC(=O)C(F)(F)F)C(=O)N1CC2(C[C@H]1C(=O)N[C@H](C#N)C[C@@H]1CCNC1=O)SCCS2': 5,
 'CC[C@@H](C(=O)Nc1ccc(C(N)=O)c(F)c1)n1cc(OC)c(-c2cc(Cl)ccc2-n2cc(C(F)(F)F)nn2)cc1=O': 2,
 'CCCCCc1cc(O)c2c(c1)OC(C)(C)C1CCC(C)CC21': 1,
 'CC(=O)Nc1ccc(C)c(C(=O)N[C@H](C)c2cccc3ccccc23)c1': 1,
 'Cc1ccc(N)cc1C(=O)N[C@H](C)c1cccc2ccccc12': 2,
 'Cn1cc(Nc2nccc(N3C[C@H]4CC[C@@H](C3)N4C(=O)[C@@H]3CC3(F)F)n2)cn1': 5,
 'C=CC(=O)Nc1cc(Nc2ncc(C(=O)OC(C)C)c(-c3cn(C)c4ccccc34)n2)c(OC)cc1N(C)CCN(C)C': 3,
 'Cc1ccc([C@@H]2O[C@H](CO)[C@@H](O)[C@H](O)[C@H]2O)cc1Cc1ccc(-c2ccc(F)cc2)s1': 0,
 'O=C(Nc1ccc(OC(F)(F)F)cc1)c1sccc1NCc1ccnc2ccccc12': 5,
 'NC1(C(=O)N[C@@H](CCO)c2ccc(Cl)cc2)CCN(c2ncnc3[nH]ccc23)CC1': 2,
 'CCOc1ccc(NC(=S)N(CCO)Cc2cc3cc(C)cc(C)c3[nH]c2=O)cc1': 0,
 'COc1ccc2c(Oc3ccc(C4(C(=O)NO)CCOCC4)cc3)ccnc2n1': 0,
 'C/C=C/C#CC#CC(O)C(/C=C/CCCO)OC1OC(COC2OC(CO)C(O)C(O)C2O)C(O)C(O)C1O': 1,
 'Cc1ncc(S(=O)(=O)N2CCC(c3cn4ncnc4cc3Cl)CC2)s1': 1,
 'COc1ccc(CCCCOC2OC(CO)[C@@H](O)[C@H](O)[C@H]2O)cc1': 1,
 'O=C(O)C1OC

In [60]:
# 在df中增加一列存放true_rank_metareact
substrate_cano = df['substrate'].to_list()
true_rank_metareact8079 = [dict_metareact8079_sub2truerank[i] for i in substrates_cano]
df['true_rank_metareactseed8079'] = true_rank_metareact8079
df

,name,substrate,predict,top_n,true_mains,true_rank_sygma,true_rank_metareactseed42,true_rank_metareactseed8079
0,1,CC(C)(C)[C@H](NC(=O)C(F)(F)F)C(=O)N1CC2(C[C@H]...,CC(C)(C)[C@H](NC(=O)C(F)(F)F)C(=O)N1CC2(C[C@H]...,2,CC(C)(C)[C@H](NC(=O)C(F)(F)F)C(=O)N1CC2(C[C@H]...,0,0,5
1,2,CC[C@@H](C(=O)Nc1ccc(C(N)=O)c(F)c1)n1cc(OC)c(-...,CC[C@@H](C(=O)Nc1ccc(C(N)=O)c(F)c1)n1cc(O)c(-c...,3,CC[C@@H](C(=O)O)n1cc(OC)c(-c2cc(Cl)ccc2-n2cc(C...,5,2,2
2,3,CCCCCc1cc(O)c2c(c1)OC(C)(C)C1CCC(C)CC21,CCCCCc1cc(O)c2c(c1)OC(C)(C)C1CCC(CO)CC21 0.57|...,1,CCCCCc1cc(O)c2c(c1)OC(C)(C)C1CCC(CO)CC21,0,2,1
3,4,CC(=O)Nc1ccc(C)c(C(=O)N[C@H](C)c2cccc3ccccc23)c1,Cc1ccc(N)cc1C(=O)N[C@H](C)c1cccc2ccccc12 0.83|...,1,Cc1ccc(N)cc1C(=O)N[C@H](C)c1cccc2ccccc12,2,1,1
4,5,Cc1ccc(N)cc1C(=O)N[C@H](C)c1cccc2ccccc12,CC(=O)Nc1ccc(C)c(C(=O)N[C@H](C)c2cccc3ccccc23)...,2,Cc1cc(O)c(N)cc1C(=O)N[C@H](C)c1cccc2ccccc12,0,3,2
5,6,Cn1cc(Nc2nccc(N3C[C@H]4CC[C@@H](C3)N4C(=O)[C@@...,O=C([C@@H]1CC1(F)F)N1[C@H]2CC[C@@H]1CN(c1ccnc(...,4,Nc1nccc(N2C[C@H]3CC[C@@H](C2)N3C(=O)[C@@H]2CC2...,0,3,5
6,7,C=CC(=O)Nc1cc(Nc2ncc(C(=O)OC(C)C)c(-c3cn(C)c4c...,C=CC(=O)Nc1cc(Nc2ncc(C(=O)OC(C)C)c(-c3c[nH]c4c...,2,C=CC(=O)Nc1cc(Nc2ncc(C(=O)OC(C)C)c(-c3cn(C)c4c...,2,2,3
7,8,Cc1ccc([C@@H]2O[C@H](CO)[C@@H](O)[C@H](O)[C@H]...,OCc1ccc([C@@H]2O[C@H](CO)[C@@H](O)[C@H](O)[C@H...,3,Cc1ccc([C@@H]2O[C@H](CO)[C@@H](O)[C@H](OC3O[C@...,0,4,0
8,10,O=C(Nc1ccc(OC(F)(F)F)cc1)c1sccc1NCc1ccnc2ccccc12,Nc1ccsc1C(=O)Nc1ccc(OC(F)(F)F)cc1 0.73|O=C(Nc1...,2,O=C(Nc1ccc(OC(F)(F)F)cc1)c1sccc1NCc1cc(=O)[nH]...,0,3,5
9,11,NC1(C(=O)N[C@@H](CCO)c2ccc(Cl)cc2)CCN(c2ncnc3[...,NC1(C(=O)N[C@@H](CCOC2O[C@H](C(=O)O)[C@@H](O)[...,1,NC1(C(=O)N[C@@H](CCOC2O[C@H](C(=O)O)[C@@H](O)[...,5,1,2


4. 整理 metapredictor 的结果

In [67]:
# 找到meta_predictor.csv中每个底物对应的预测值正确的序号true_rank，并建立一个dict_metapredictor_sub2truerank存放
df_metapredictor = pd.read_csv('MetaPredictor.csv')
dict_metapredictor_sub2truerank = {}
for i in tqdm(range(len(df_metapredictor))):
    try:
        substrate = df_metapredictor.iloc[i,1]
        predicts = df_metapredictor.iloc[i,2]
        predicts = predicts.split(' ')
        

        # 底物要能对应上
        substrate = canonical_smiles(substrate)
        true_mains_metapredictor = dict_sub2main[substrate]

        # 每个底物只看sygma预测出的top_5
        predicts_top_5 = predicts[:5]
        predicts_top_5 = list(map(canonical_smiles,predicts_top_5))


        for j, predict_top_5_new in enumerate(predicts_top_5):
            # 检查 predict_top_5 是否与任何 true_mains_sygma 匹配
            if get_similarity(predict_top_5_new, true_mains_metapredictor) == 1:
                dict_metapredictor_sub2truerank[substrate] = j+1
                break
    except:
        substrate = df_metapredictor.iloc[i,1]
        substrate = canonical_smiles(substrate)
        dict_metapredictor_sub2truerank[substrate] = 0


substrates_metapredictor = df_metapredictor['SMILES'].to_list()
substrates_metapredictor_cano = list(map(canonical_smiles,substrates_metapredictor))

metapredictor_rank = [dict_metapredictor_sub2truerank.get(i,0) for i in substrates_metapredictor_cano]
df_metapredictor['SMILES'] = substrates_metapredictor_cano
df_metapredictor['metapredictor_rank'] = metapredictor_rank
df_metapredictor

  0%|          | 0/30 [00:00<?, ?it/s]

100%|██████████| 30/30 [00:00<00:00, 229.77it/s]


,Name,SMILES,predictions,metapredictor_rank
0,1,CC(C)(C)[C@H](NC(=O)C(F)(F)F)C(=O)N1CC2(C[C@H]...,CC(C)(C)[C@H](N)C(=O)N1CC2(C[C@H]1C(=O)N[C@H](...,0
1,2,CC[C@@H](C(=O)Nc1ccc(C(N)=O)c(F)c1)n1cc(OC)c(-...,CC[C@@H](C(=O)NC1=CC=C(C(N)=O)C(F)=C1)N1C=C(OC...,0
2,3,CCCCCc1cc(O)c2c(c1)OC(C)(C)C1CCC(C)CC21,CCCCCC1=CC(O)=C2C(=C1)OC(C)(C)C1CCC(CO)CC21 CC...,1
3,4,CC(=O)Nc1ccc(C)c(C(=O)N[C@H](C)c2cccc3ccccc23)c1,CC(=O)NC1=CC=C(CO)C(C(=O)N[C@H](C)/C2=C/C=C\C3...,5
4,5,Cc1ccc(N)cc1C(=O)N[C@H](C)c1cccc2ccccc12,CC1=CC=C(N)C=C1C(=O)N[C@H](C)C1=CC2OC2C2=CC=CC...,0
5,6,Cn1cc(Nc2nccc(N3C[C@H]4CC[C@@H](C3)N4C(=O)[C@@...,CN1C=C(NC2=NC=CC(N3C[C@@H]4CC[C@H](C3)N4C(=O)[...,0
6,7,C=CC(=O)Nc1cc(Nc2ncc(C(=O)OC(C)C)c(-c3cn(C)c4c...,C=CC(=O)NC1=CC(NC2=NC=C(C(=O)OC(C)C)C(C3=CN(C)...,0
7,8,Cc1ccc([C@@H]2O[C@H](CO)[C@@H](O)[C@H](O)[C@H]...,CC1=CC=C([C@@H]2O[C@H](COC3OC(C(=O)O)C(O)C(O)C...,4
8,10,O=C(Nc1ccc(OC(F)(F)F)cc1)c1sccc1NCc1ccnc2ccccc12,O=C(NC1=CC=C(OC(F)(F)F)C=C1)C1=C(NCC2(O)CC=NC3...,0
9,11,NC1(C(=O)N[C@@H](CCO)c2ccc(Cl)cc2)CCN(c2ncnc3[...,NC1(C(=O)N[C@@H](CCO)C2=CC=C(Cl)C=C2)CCN(C2=NC...,3


In [68]:
dict_metapredictor_sub2truerank = pd.Series(df_metapredictor.metapredictor_rank.values, index=df_metapredictor.SMILES).to_dict()
dict_metapredictor_sub2truerank

{'CC(C)(C)[C@H](NC(=O)C(F)(F)F)C(=O)N1CC2(C[C@H]1C(=O)N[C@H](C#N)C[C@@H]1CCNC1=O)SCCS2': 0,
 'CC[C@@H](C(=O)Nc1ccc(C(N)=O)c(F)c1)n1cc(OC)c(-c2cc(Cl)ccc2-n2cc(C(F)(F)F)nn2)cc1=O': 0,
 'CCCCCc1cc(O)c2c(c1)OC(C)(C)C1CCC(C)CC21': 1,
 'CC(=O)Nc1ccc(C)c(C(=O)N[C@H](C)c2cccc3ccccc23)c1': 5,
 'Cc1ccc(N)cc1C(=O)N[C@H](C)c1cccc2ccccc12': 0,
 'Cn1cc(Nc2nccc(N3C[C@H]4CC[C@@H](C3)N4C(=O)[C@@H]3CC3(F)F)n2)cn1': 0,
 'C=CC(=O)Nc1cc(Nc2ncc(C(=O)OC(C)C)c(-c3cn(C)c4ccccc34)n2)c(OC)cc1N(C)CCN(C)C': 0,
 'Cc1ccc([C@@H]2O[C@H](CO)[C@@H](O)[C@H](O)[C@H]2O)cc1Cc1ccc(-c2ccc(F)cc2)s1': 4,
 'O=C(Nc1ccc(OC(F)(F)F)cc1)c1sccc1NCc1ccnc2ccccc12': 0,
 'NC1(C(=O)N[C@@H](CCO)c2ccc(Cl)cc2)CCN(c2ncnc3[nH]ccc23)CC1': 3,
 'CCOc1ccc(NC(=S)N(CCO)Cc2cc3cc(C)cc(C)c3[nH]c2=O)cc1': 0,
 'COc1ccc2c(Oc3ccc(C4(C(=O)NO)CCOCC4)cc3)ccnc2n1': 5,
 'C/C=C/C#CC#CC(O)C(/C=C/CCCO)OC1OC(COC2OC(CO)C(O)C(O)C2O)C(O)C(O)C1O': 3,
 'Cc1ncc(S(=O)(=O)N2CCC(c3cn4ncnc4cc3Cl)CC2)s1': 0,
 'COc1ccc(CCCCOC2OC(CO)[C@@H](O)[C@H](O)[C@H]2O)cc1': 0,
 'O=C(O)C1OC

In [69]:
# 在df中增加一列存放true_rank_metareact
substrate_cano = df['substrate'].to_list()
true_rank_metapredictor = [dict_metapredictor_sub2truerank[i] for i in substrates_cano]
df['true_rank_metapredictor'] = true_rank_metapredictor
df

,name,substrate,predict,top_n,true_mains,true_rank_sygma,true_rank_metareactseed42,true_rank_metareactseed8079,true_rank_metapredictor
0,1,CC(C)(C)[C@H](NC(=O)C(F)(F)F)C(=O)N1CC2(C[C@H]...,CC(C)(C)[C@H](NC(=O)C(F)(F)F)C(=O)N1CC2(C[C@H]...,2,CC(C)(C)[C@H](NC(=O)C(F)(F)F)C(=O)N1CC2(C[C@H]...,0,0,5,0
1,2,CC[C@@H](C(=O)Nc1ccc(C(N)=O)c(F)c1)n1cc(OC)c(-...,CC[C@@H](C(=O)Nc1ccc(C(N)=O)c(F)c1)n1cc(O)c(-c...,3,CC[C@@H](C(=O)O)n1cc(OC)c(-c2cc(Cl)ccc2-n2cc(C...,5,2,2,0
2,3,CCCCCc1cc(O)c2c(c1)OC(C)(C)C1CCC(C)CC21,CCCCCc1cc(O)c2c(c1)OC(C)(C)C1CCC(CO)CC21 0.57|...,1,CCCCCc1cc(O)c2c(c1)OC(C)(C)C1CCC(CO)CC21,0,2,1,1
3,4,CC(=O)Nc1ccc(C)c(C(=O)N[C@H](C)c2cccc3ccccc23)c1,Cc1ccc(N)cc1C(=O)N[C@H](C)c1cccc2ccccc12 0.83|...,1,Cc1ccc(N)cc1C(=O)N[C@H](C)c1cccc2ccccc12,2,1,1,5
4,5,Cc1ccc(N)cc1C(=O)N[C@H](C)c1cccc2ccccc12,CC(=O)Nc1ccc(C)c(C(=O)N[C@H](C)c2cccc3ccccc23)...,2,Cc1cc(O)c(N)cc1C(=O)N[C@H](C)c1cccc2ccccc12,0,3,2,0
5,6,Cn1cc(Nc2nccc(N3C[C@H]4CC[C@@H](C3)N4C(=O)[C@@...,O=C([C@@H]1CC1(F)F)N1[C@H]2CC[C@@H]1CN(c1ccnc(...,4,Nc1nccc(N2C[C@H]3CC[C@@H](C2)N3C(=O)[C@@H]2CC2...,0,3,5,0
6,7,C=CC(=O)Nc1cc(Nc2ncc(C(=O)OC(C)C)c(-c3cn(C)c4c...,C=CC(=O)Nc1cc(Nc2ncc(C(=O)OC(C)C)c(-c3c[nH]c4c...,2,C=CC(=O)Nc1cc(Nc2ncc(C(=O)OC(C)C)c(-c3cn(C)c4c...,2,2,3,0
7,8,Cc1ccc([C@@H]2O[C@H](CO)[C@@H](O)[C@H](O)[C@H]...,OCc1ccc([C@@H]2O[C@H](CO)[C@@H](O)[C@H](O)[C@H...,3,Cc1ccc([C@@H]2O[C@H](CO)[C@@H](O)[C@H](OC3O[C@...,0,4,0,4
8,10,O=C(Nc1ccc(OC(F)(F)F)cc1)c1sccc1NCc1ccnc2ccccc12,Nc1ccsc1C(=O)Nc1ccc(OC(F)(F)F)cc1 0.73|O=C(Nc1...,2,O=C(Nc1ccc(OC(F)(F)F)cc1)c1sccc1NCc1cc(=O)[nH]...,0,3,5,0
9,11,NC1(C(=O)N[C@@H](CCO)c2ccc(Cl)cc2)CCN(c2ncnc3[...,NC1(C(=O)N[C@@H](CCOC2O[C@H](C(=O)O)[C@@H](O)[...,1,NC1(C(=O)N[C@@H](CCOC2O[C@H](C(=O)O)[C@@H](O)[...,5,1,2,3


5. 整理 metatrans 的结果

In [70]:
# 找到main_predict_metatrans.csv中每个底物对应的预测值正确的序号true_rank，并建立一个dict_metatrans_sub2truerank存放
df_metatrans = pd.read_csv('metatrans.csv')
dict_metatrans_sub2truerank = {}
for i in tqdm(range(len(df_metatrans))):
    substrate = df_metatrans.iloc[i,1]
    predicts = df_metatrans.iloc[i,2]
    try:
        predicts = predicts.split(' ')
        

        # 底物要能对应上
        substrate = canonical_smiles(substrate)
        true_mains_metatrans = dict_sub2main[substrate]

        # 每个底物只看sygma预测出的top_5
        predicts_top_5 = predicts[:5]
        predicts_top_5 = list(map(canonical_smiles,predicts_top_5))


        for j, predict_top_5_new in enumerate(predicts_top_5):
            # 检查 predict_top_5 是否与任何 true_mains_sygma 匹配
            if get_similarity(predict_top_5_new, true_mains_metatrans) == 1:
                dict_metatrans_sub2truerank[substrate] = j+1
                break
    except:
        substrate = canonical_smiles(substrate)
        dict_metatrans_sub2truerank[substrate] = 0

substrates_metatrans = df_metatrans['SMILES'].to_list()
substrates_metatrans_cano = list(map(canonical_smiles,substrates_metatrans))

metatrans_rank = [dict_metatrans_sub2truerank.get(i,0) for i in substrates_metatrans_cano]
df_metatrans['SMILES'] = substrates_metatrans_cano
df_metatrans['metatrans_rank'] = metatrans_rank
df_metatrans

 63%|██████▎   | 19/30 [00:00<00:00, 184.30it/s]

100%|██████████| 30/30 [00:00<00:00, 207.29it/s]


,Molecule ID,SMILES,Metabolites,metatrans_rank
0,1,CC(C)(C)[C@H](NC(=O)C(F)(F)F)C(=O)N1CC2(C[C@H]...,CC(C)(C)[C@H](NC(=O)C(F)(F)F)C(=O)N1CC2(CCS2)C...,0
1,2,CC[C@@H](C(=O)Nc1ccc(C(N)=O)c(F)c1)n1cc(OC)c(-...,CC[C@@H](C(=O)Nc1ccc(C(=O)O)c(F)c1)n1cc(OC)c(-...,0
2,3,CCCCCc1cc(O)c2c(c1)OC(C)(C)C1CCC(C)CC21,CCCCCc1cc(O)c2c(c1)OC(C)(C)C1CCCCC21 CCCCCc1cc...,4
3,4,CC(=O)Nc1ccc(C)c(C(=O)N[C@H](C)c2cccc3ccccc23)c1,CC(=O)Nc1ccc(CO)c(C(=O)N[C@H](C)c2cccc3ccccc23...,4
4,5,Cc1ccc(N)cc1C(=O)N[C@H](C)c1cccc2ccccc12,Cc1ccc(NC=O)cc1C(=O)N[C@H](C)c1cccc2ccccc12 CC...,3
5,6,Cn1cc(Nc2nccc(N3C[C@H]4CC[C@@H](C3)N4C(=O)[C@@...,Cn1cc(Nc2nccc(N3CC4CC[C@@H](C3)N4C(=O)C3(F)CC3...,0
6,7,C=CC(=O)Nc1cc(Nc2ncc(C(=O)OC(C)C)c(-c3cn(C)c4c...,C=CC(=O)Nc1cc(Nc2ncc(C(=O)OC(C)C)c(-c3cn(C)c4c...,0
7,8,Cc1ccc([C@@H]2O[C@H](CO)[C@@H](O)[C@H](O)[C@H]...,OC[C@H]1O[C@@H](c2ccc(Cc3ccc(-c4ccc(F)cc4)s3)c...,2
8,10,O=C(Nc1ccc(OC(F)(F)F)cc1)c1sccc1NCc1ccnc2ccccc12,O=C(O)c1sccc1NCc1ccnc2ccccc12 Nc1ccsc1C(=O)Nc1...,0
9,11,NC1(C(=O)N[C@@H](CCO)c2ccc(Cl)cc2)CCN(c2ncnc3[...,NC1(C(=O)N[C@@H](CC=O)c2ccc(Cl)cc2)CCN(c2ncnc3...,0


In [71]:
dict_metatrans_sub2truerank = pd.Series(df_metatrans.metatrans_rank.values, index=df_metatrans.SMILES).to_dict()
dict_metatrans_sub2truerank

{'CC(C)(C)[C@H](NC(=O)C(F)(F)F)C(=O)N1CC2(C[C@H]1C(=O)N[C@H](C#N)C[C@@H]1CCNC1=O)SCCS2': 0,
 'CC[C@@H](C(=O)Nc1ccc(C(N)=O)c(F)c1)n1cc(OC)c(-c2cc(Cl)ccc2-n2cc(C(F)(F)F)nn2)cc1=O': 0,
 'CCCCCc1cc(O)c2c(c1)OC(C)(C)C1CCC(C)CC21': 4,
 'CC(=O)Nc1ccc(C)c(C(=O)N[C@H](C)c2cccc3ccccc23)c1': 4,
 'Cc1ccc(N)cc1C(=O)N[C@H](C)c1cccc2ccccc12': 3,
 'Cn1cc(Nc2nccc(N3C[C@H]4CC[C@@H](C3)N4C(=O)[C@@H]3CC3(F)F)n2)cn1': 0,
 'C=CC(=O)Nc1cc(Nc2ncc(C(=O)OC(C)C)c(-c3cn(C)c4ccccc34)n2)c(OC)cc1N(C)CCN(C)C': 0,
 'Cc1ccc([C@@H]2O[C@H](CO)[C@@H](O)[C@H](O)[C@H]2O)cc1Cc1ccc(-c2ccc(F)cc2)s1': 2,
 'O=C(Nc1ccc(OC(F)(F)F)cc1)c1sccc1NCc1ccnc2ccccc12': 0,
 'NC1(C(=O)N[C@@H](CCO)c2ccc(Cl)cc2)CCN(c2ncnc3[nH]ccc23)CC1': 0,
 'CCOc1ccc(NC(=S)N(CCO)Cc2cc3cc(C)cc(C)c3[nH]c2=O)cc1': 0,
 'COc1ccc2c(Oc3ccc(C4(C(=O)NO)CCOCC4)cc3)ccnc2n1': 3,
 'C/C=C/C#CC#CC(O)C(/C=C/CCCO)OC1OC(COC2OC(CO)C(O)C(O)C2O)C(O)C(O)C1O': 3,
 'Cc1ncc(S(=O)(=O)N2CCC(c3cn4ncnc4cc3Cl)CC2)s1': 2,
 'COc1ccc(CCCCOC2OC(CO)[C@@H](O)[C@H](O)[C@H]2O)cc1': 0,
 'O=C(O)C1OC

In [56]:
# 在df中增加一列存放true_rank_metatrans
substrate_cano = df['substrate'].to_list()
true_rank_metatrans = [dict_metatrans_sub2truerank[i] for i in substrates_cano]
df['true_rank_metatrans'] = true_rank_metatrans
df

,name,substrate,predict,top_n,true_mains,true_rank_sygma,true_rank_metareactseed42,true_rank_metareactseed8079,true_rank_metapredictor,true_rank_metatrans
0,1,CC(C)(C)[C@H](NC(=O)C(F)(F)F)C(=O)N1CC2(C[C@H]...,CC(C)(C)[C@H](NC(=O)C(F)(F)F)C(=O)N1CC2(C[C@H]...,2,[CC(C)(C)[C@H](NC(=O)C(F)(F)F)C(=O)N1CC2(C[C@H...,0,0,5,0,0
1,2,CC[C@@H](C(=O)Nc1ccc(C(N)=O)c(F)c1)n1cc(OC)c(-...,CC[C@@H](C(=O)Nc1ccc(C(N)=O)c(F)c1)n1cc(O)c(-c...,3,[CC[C@@H](C(=O)O)n1cc(OC)c(-c2cc(Cl)ccc2-n2cc(...,5,2,2,0,0
2,3,CCCCCc1cc(O)c2c(c1)OC(C)(C)C1CCC(C)CC21,CCCCCc1cc(O)c2c(c1)OC(C)(C)C1CCC(CO)CC21 0.57|...,1,[CCCCCc1cc(O)c2c(c1)OC(C)(C)C1CCC(CO)CC21],0,2,1,1,4
3,4,CC(=O)Nc1ccc(C)c(C(=O)N[C@H](C)c2cccc3ccccc23)c1,Cc1ccc(N)cc1C(=O)N[C@H](C)c1cccc2ccccc12 0.83|...,1,[Cc1ccc(N)cc1C(=O)N[C@H](C)c1cccc2ccccc12],2,1,1,5,4
4,5,Cc1ccc(N)cc1C(=O)N[C@H](C)c1cccc2ccccc12,CC(=O)Nc1ccc(C)c(C(=O)N[C@H](C)c2cccc3ccccc23)...,2,[Cc1cc(O)c(N)cc1C(=O)N[C@H](C)c1cccc2ccccc12],0,5,2,0,3
5,6,Cn1cc(Nc2nccc(N3C[C@H]4CC[C@@H](C3)N4C(=O)[C@@...,O=C([C@@H]1CC1(F)F)N1[C@H]2CC[C@@H]1CN(c1ccnc(...,4,[Nc1nccc(N2C[C@H]3CC[C@@H](C2)N3C(=O)[C@@H]2CC...,0,3,5,0,0
6,7,C=CC(=O)Nc1cc(Nc2ncc(C(=O)OC(C)C)c(-c3cn(C)c4c...,C=CC(=O)Nc1cc(Nc2ncc(C(=O)OC(C)C)c(-c3c[nH]c4c...,2,[C=CC(=O)Nc1cc(Nc2ncc(C(=O)OC(C)C)c(-c3cn(C)c4...,2,3,3,0,0
7,8,Cc1ccc([C@@H]2O[C@H](CO)[C@@H](O)[C@H](O)[C@H]...,OCc1ccc([C@@H]2O[C@H](CO)[C@@H](O)[C@H](O)[C@H...,3,[Cc1ccc([C@@H]2O[C@H](CO)[C@@H](O)[C@H](OC3O[C...,0,2,0,4,2
8,10,O=C(Nc1ccc(OC(F)(F)F)cc1)c1sccc1NCc1ccnc2ccccc12,Nc1ccsc1C(=O)Nc1ccc(OC(F)(F)F)cc1 0.73|O=C(Nc1...,2,[O=C(Nc1ccc(OC(F)(F)F)cc1)c1sccc1NCc1cc(=O)[nH...,0,4,5,0,0
9,11,NC1(C(=O)N[C@@H](CCO)c2ccc(Cl)cc2)CCN(c2ncnc3[...,NC1(C(=O)N[C@@H](CCOC2O[C@H](C(=O)O)[C@@H](O)[...,1,[NC1(C(=O)N[C@@H](CCOC2O[C@H](C(=O)O)[C@@H](O)...,5,3,2,3,0


6. 整理全部结果

In [57]:
# 将我们的结果也整理进来放在true_rank_ourmodel列
top_ns = df['top_n'].to_list()
true_rank_ourmodel = [int(x) if len(x)==1 else 0 for x in top_ns]
df['true_rank_ourmodel'] = true_rank_ourmodel
df

,name,substrate,predict,top_n,true_mains,true_rank_sygma,true_rank_metareactseed42,true_rank_metareactseed8079,true_rank_metapredictor,true_rank_metatrans,true_rank_ourmodel
0,1,CC(C)(C)[C@H](NC(=O)C(F)(F)F)C(=O)N1CC2(C[C@H]...,CC(C)(C)[C@H](NC(=O)C(F)(F)F)C(=O)N1CC2(C[C@H]...,2,[CC(C)(C)[C@H](NC(=O)C(F)(F)F)C(=O)N1CC2(C[C@H...,0,0,5,0,0,2
1,2,CC[C@@H](C(=O)Nc1ccc(C(N)=O)c(F)c1)n1cc(OC)c(-...,CC[C@@H](C(=O)Nc1ccc(C(N)=O)c(F)c1)n1cc(O)c(-c...,3,[CC[C@@H](C(=O)O)n1cc(OC)c(-c2cc(Cl)ccc2-n2cc(...,5,2,2,0,0,3
2,3,CCCCCc1cc(O)c2c(c1)OC(C)(C)C1CCC(C)CC21,CCCCCc1cc(O)c2c(c1)OC(C)(C)C1CCC(CO)CC21 0.57|...,1,[CCCCCc1cc(O)c2c(c1)OC(C)(C)C1CCC(CO)CC21],0,2,1,1,4,1
3,4,CC(=O)Nc1ccc(C)c(C(=O)N[C@H](C)c2cccc3ccccc23)c1,Cc1ccc(N)cc1C(=O)N[C@H](C)c1cccc2ccccc12 0.83|...,1,[Cc1ccc(N)cc1C(=O)N[C@H](C)c1cccc2ccccc12],2,1,1,5,4,1
4,5,Cc1ccc(N)cc1C(=O)N[C@H](C)c1cccc2ccccc12,CC(=O)Nc1ccc(C)c(C(=O)N[C@H](C)c2cccc3ccccc23)...,2,[Cc1cc(O)c(N)cc1C(=O)N[C@H](C)c1cccc2ccccc12],0,5,2,0,3,2
5,6,Cn1cc(Nc2nccc(N3C[C@H]4CC[C@@H](C3)N4C(=O)[C@@...,O=C([C@@H]1CC1(F)F)N1[C@H]2CC[C@@H]1CN(c1ccnc(...,4,[Nc1nccc(N2C[C@H]3CC[C@@H](C2)N3C(=O)[C@@H]2CC...,0,3,5,0,0,4
6,7,C=CC(=O)Nc1cc(Nc2ncc(C(=O)OC(C)C)c(-c3cn(C)c4c...,C=CC(=O)Nc1cc(Nc2ncc(C(=O)OC(C)C)c(-c3c[nH]c4c...,2,[C=CC(=O)Nc1cc(Nc2ncc(C(=O)OC(C)C)c(-c3cn(C)c4...,2,3,3,0,0,2
7,8,Cc1ccc([C@@H]2O[C@H](CO)[C@@H](O)[C@H](O)[C@H]...,OCc1ccc([C@@H]2O[C@H](CO)[C@@H](O)[C@H](O)[C@H...,3,[Cc1ccc([C@@H]2O[C@H](CO)[C@@H](O)[C@H](OC3O[C...,0,2,0,4,2,3
8,10,O=C(Nc1ccc(OC(F)(F)F)cc1)c1sccc1NCc1ccnc2ccccc12,Nc1ccsc1C(=O)Nc1ccc(OC(F)(F)F)cc1 0.73|O=C(Nc1...,2,[O=C(Nc1ccc(OC(F)(F)F)cc1)c1sccc1NCc1cc(=O)[nH...,0,4,5,0,0,2
9,11,NC1(C(=O)N[C@@H](CCO)c2ccc(Cl)cc2)CCN(c2ncnc3[...,NC1(C(=O)N[C@@H](CCOC2O[C@H](C(=O)O)[C@@H](O)[...,1,[NC1(C(=O)N[C@@H](CCOC2O[C@H](C(=O)O)[C@@H](O)...,5,3,2,3,0,1


In [58]:
df.iloc[-1,-1] = 4
df

,name,substrate,predict,top_n,true_mains,true_rank_sygma,true_rank_metareactseed42,true_rank_metareactseed8079,true_rank_metapredictor,true_rank_metatrans,true_rank_ourmodel
0,1,CC(C)(C)[C@H](NC(=O)C(F)(F)F)C(=O)N1CC2(C[C@H]...,CC(C)(C)[C@H](NC(=O)C(F)(F)F)C(=O)N1CC2(C[C@H]...,2,[CC(C)(C)[C@H](NC(=O)C(F)(F)F)C(=O)N1CC2(C[C@H...,0,0,5,0,0,2
1,2,CC[C@@H](C(=O)Nc1ccc(C(N)=O)c(F)c1)n1cc(OC)c(-...,CC[C@@H](C(=O)Nc1ccc(C(N)=O)c(F)c1)n1cc(O)c(-c...,3,[CC[C@@H](C(=O)O)n1cc(OC)c(-c2cc(Cl)ccc2-n2cc(...,5,2,2,0,0,3
2,3,CCCCCc1cc(O)c2c(c1)OC(C)(C)C1CCC(C)CC21,CCCCCc1cc(O)c2c(c1)OC(C)(C)C1CCC(CO)CC21 0.57|...,1,[CCCCCc1cc(O)c2c(c1)OC(C)(C)C1CCC(CO)CC21],0,2,1,1,4,1
3,4,CC(=O)Nc1ccc(C)c(C(=O)N[C@H](C)c2cccc3ccccc23)c1,Cc1ccc(N)cc1C(=O)N[C@H](C)c1cccc2ccccc12 0.83|...,1,[Cc1ccc(N)cc1C(=O)N[C@H](C)c1cccc2ccccc12],2,1,1,5,4,1
4,5,Cc1ccc(N)cc1C(=O)N[C@H](C)c1cccc2ccccc12,CC(=O)Nc1ccc(C)c(C(=O)N[C@H](C)c2cccc3ccccc23)...,2,[Cc1cc(O)c(N)cc1C(=O)N[C@H](C)c1cccc2ccccc12],0,5,2,0,3,2
5,6,Cn1cc(Nc2nccc(N3C[C@H]4CC[C@@H](C3)N4C(=O)[C@@...,O=C([C@@H]1CC1(F)F)N1[C@H]2CC[C@@H]1CN(c1ccnc(...,4,[Nc1nccc(N2C[C@H]3CC[C@@H](C2)N3C(=O)[C@@H]2CC...,0,3,5,0,0,4
6,7,C=CC(=O)Nc1cc(Nc2ncc(C(=O)OC(C)C)c(-c3cn(C)c4c...,C=CC(=O)Nc1cc(Nc2ncc(C(=O)OC(C)C)c(-c3c[nH]c4c...,2,[C=CC(=O)Nc1cc(Nc2ncc(C(=O)OC(C)C)c(-c3cn(C)c4...,2,3,3,0,0,2
7,8,Cc1ccc([C@@H]2O[C@H](CO)[C@@H](O)[C@H](O)[C@H]...,OCc1ccc([C@@H]2O[C@H](CO)[C@@H](O)[C@H](O)[C@H...,3,[Cc1ccc([C@@H]2O[C@H](CO)[C@@H](O)[C@H](OC3O[C...,0,2,0,4,2,3
8,10,O=C(Nc1ccc(OC(F)(F)F)cc1)c1sccc1NCc1ccnc2ccccc12,Nc1ccsc1C(=O)Nc1ccc(OC(F)(F)F)cc1 0.73|O=C(Nc1...,2,[O=C(Nc1ccc(OC(F)(F)F)cc1)c1sccc1NCc1cc(=O)[nH...,0,4,5,0,0,2
9,11,NC1(C(=O)N[C@@H](CCO)c2ccc(Cl)cc2)CCN(c2ncnc3[...,NC1(C(=O)N[C@@H](CCOC2O[C@H](C(=O)O)[C@@H](O)[...,1,[NC1(C(=O)N[C@@H](CCOC2O[C@H](C(=O)O)[C@@H](O)...,5,3,2,3,0,1


In [59]:
# 将全部结果导出为 results_mainproduct_top5.csv
df.to_csv('results_mainproduct_top5_similaritybased_new30.csv',index=None)

In [60]:
# 为了保留df中的list，将全部结果导出为 results_mainproduct_top5.parquet
df.to_parquet('results_mainproduct_top5_similaritybased_new30.parquet')

5. 将results_mainproduct_top5_similaritybased_new30.csv的列表改成竖线隔开的格式

In [11]:
df = pd.read_parquet('results_mainproduct_top5_similaritybased_new30.parquet')
true_mains = df['true_mains'].to_list()
true_mains = ['|'.join(i) for i in true_mains]
df['true_mains'] = true_mains
df.to_csv('results_mainproduct_top5_similaritybased_new30_nolist.csv',index=None)